# 01 — SQL Game Analysis

In the previous notebook, I tested whether the Lichess dataset was usable for analysis. I checked the row structure, removed raw index artifact columns, inspected missing values and duplicates, and confirmed that the main game-level metadata is reliable enough to continue.

This notebook starts the actual analysis stage. The goal is to use SQL to answer focused questions about online chess games, especially around time controls, rating differences, game results, and termination types.

Instead of only asking what happened, I want to practice asking deeper analytical questions:

- How do faster time controls change the way games end?
- Are time forfeits more common in Bullet and Blitz games?
- Does rating advantage explain who wins?
- Are upsets more common in faster time controls?
- Do different openings or categories show different outcome patterns?

This notebook will focus on game-level analysis first. More complex move-level, clock, or engine-evaluation features can be explored later if they add value.


## 1. Load the Clean Game-Level Data into SQLite

The previous notebook exported a clean game-level dataset after completing the feasibility and data quality checks. Before writing SQL queries, I need to load that processed CSV into a SQLite database.

Opening a SQLite connection only creates or connects to the database file. It does not automatically create tables. Therefore, this section loads the processed CSV with Pandas, writes it into SQLite as a table called `games`, and verifies that the table was created correctly.


In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DB_PATH = PROCESSED_DIR / "games.db"

PROCESSED_FILE = PROCESSED_DIR / "games_clean.csv"

games_df = pd.read_csv(PROCESSED_FILE)

conn = sqlite3.connect(DB_PATH)

def run_query(query):
    return pd.read_sql_query(query, conn)

games_df.to_sql("games", conn, if_exists="replace", index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


In [2]:
games_df.shape

(200000, 21)

### Verify SQLite Table

Quick database check before starting the analysis.

In [3]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS total_games
FROM games;
""",
conn
)


,total_games
0,200000


The SQL row count matches the processed CSV row count, confirming that the cleaned game-level dataset was loaded into SQLite successfully.

## 2. Termination Patterns by Game Category

The first analysis question is whether faster chess categories are more likely to end by time forfeit.

To answer this, I group games by category and calculate the total number of games, the number of normal endings, the number of time forfeits, and the time-forfeit rate for each category.

In [4]:
termination_patterns_category_query = """
WITH termination_by_category AS (
	SELECT
		g.category,
		COUNT(*) AS total_games,
		SUM(CASE
				WHEN g.Termination = 'Normal' THEN 1
				ELSE 0
			END) AS normal_games,
		SUM(CASE
				WHEN g.Termination = 'Time forfeit' THEN 1
				ELSE 0
			END) AS time_forfeit_games
	FROM games AS g
	GROUP BY g.category
)

SELECT
	tbc.category,
	tbc.total_games,
	tbc.normal_games,
	tbc.time_forfeit_games,
	ROUND(tbc.time_forfeit_games * 100.0 / tbc.total_games, 2) AS time_forfeit_rate
FROM termination_by_category AS tbc
ORDER BY tbc.total_games DESC;
"""

termination_patterns_category_df = run_query(termination_patterns_category_query)

termination_patterns_category_df


,category,total_games,normal_games,time_forfeit_games,time_forfeit_rate
0,Blitz,86333,67554,18778,21.75
1,Rapid,45216,39235,5973,13.21
2,Bullet,43253,21758,21495,49.70
3,Classical,25198,22970,2220,8.81


Bullet has the highest time-forfeit rate by a large margin. Almost half of Bullet games ended by time forfeit, compared with much lower rates in Blitz, Rapid, and Classical games.

Blitz is the most common category in this dataset, but Bullet is the category most strongly affected by clock pressure. This supports the idea that faster game formats change how games end, especially by increasing the chance of a time-forfeit ending.

## 3. Time-Forfeit Wins by Color

The previous section showed that faster game categories, especially Bullet, have much higher time-forfeit rates.

The next question is whether those time-forfeit endings favor one color. If White or Black wins much more often in time-forfeit games, then the clock-pressure pattern could also be connected to color advantage. If the split is balanced, then time pressure affects how games end, but not necessarily which color benefits from those endings.

To answer this, I filter to decisive time-forfeit games and compare how often White and Black win within each category.


In [5]:
time_forfeit_wins_by_color_query = """
WITH time_forfeit_wins AS (
    SELECT
        Category,
        COUNT(*) AS time_forfeit_wins,
        SUM(CASE
                WHEN winner = 'White' THEN 1
                ELSE 0
            END) AS white_time_forfeit_wins,
        SUM(CASE
                WHEN winner = 'Black' THEN 1
                ELSE 0
            END) AS black_time_forfeit_wins
    FROM games
    WHERE Termination = 'Time forfeit'
      AND winner IN ('White', 'Black')
    GROUP BY Category
)

SELECT
    Category,
    time_forfeit_wins,
    white_time_forfeit_wins,
    black_time_forfeit_wins,
    ROUND(white_time_forfeit_wins * 100.0 / time_forfeit_wins, 2) AS white_win_percentage,
    ROUND(black_time_forfeit_wins * 100.0 / time_forfeit_wins, 2) AS black_win_percentage
FROM time_forfeit_wins
ORDER BY time_forfeit_wins DESC;
"""

time_forfeit_wins_by_color_df = run_query(time_forfeit_wins_by_color_query)

time_forfeit_wins_by_color_df


,Category,time_forfeit_wins,white_time_forfeit_wins,black_time_forfeit_wins,white_win_percentage,black_win_percentage
0,Bullet,21435,10642,10793,49.65,50.35
1,Blitz,18609,9321,9288,50.09,49.91
2,Rapid,5878,2946,2932,50.12,49.88
3,Classical,2199,1083,1116,49.25,50.75


Among decisive time-forfeit games, wins are almost evenly split between White and Black across all categories. This suggests that although faster categories have many more time-forfeit endings, those time forfeits do not strongly favor either color.

This supports the idea that clock pressure changes how games end, especially in Bullet, but does not appear to create a major White-versus-Black advantage in time-forfeit outcomes.


## 4. Rating Favorite Performance

The previous section showed that time-forfeit wins are fairly balanced between White and Black, so the higher time-forfeit rate in faster games does not appear to create a major color advantage.

The next question is whether rating advantage explains game outcomes. In the feasibility notebook, I created a `rating_favorite` feature using a 50-point Elo threshold. This separates games into `White favorite`, `Black favorite`, and `Close rating`.

In this section, I check how often the rating favorite wins overall before comparing that pattern across game categories.

In [6]:
rating_favorite_performance_query = """
WITH favorite_outcomes AS (
    SELECT
        rating_favorite,
        CASE
            WHEN rating_favorite = 'White favorite' AND winner = 'White' THEN 'Favorite won'
            WHEN rating_favorite = 'Black favorite' AND winner = 'Black' THEN 'Favorite won'
            ELSE 'Upset'
        END AS outcome
    FROM games
    WHERE rating_favorite IN ('White favorite', 'Black favorite')
      AND winner IN ('White', 'Black')
),

outcome_counts AS (
    SELECT
        rating_favorite,
        outcome,
        COUNT(*) AS games
    FROM favorite_outcomes
    GROUP BY rating_favorite, outcome
),

favorite_totals AS (
    SELECT
        rating_favorite,
        SUM(games) AS total_games
    FROM outcome_counts
    GROUP BY rating_favorite
)

SELECT
    oc.rating_favorite,
    oc.outcome,
    oc.games,
    ROUND(oc.games * 100.0 / ft.total_games, 2) AS percentage_within_favorite
FROM outcome_counts AS oc
JOIN favorite_totals AS ft
    ON oc.rating_favorite = ft.rating_favorite
ORDER BY
    oc.rating_favorite,
    oc.outcome;
"""

rating_favorite_performance_df = run_query(rating_favorite_performance_query)

rating_favorite_performance_df


,rating_favorite,outcome,games,percentage_within_favorite
0,Black favorite,Favorite won,26556,60.06
1,Black favorite,Upset,17658,39.94
2,White favorite,Favorite won,27449,62.46
3,White favorite,Upset,16496,37.54


Across decisive games with a clear rating favorite, the favorite wins about 61% of the time. This suggests that rating advantage is meaningful, but not overwhelming. Upsets still happen in roughly 39% of decisive favorite-vs-underdog games.

## 5. Rating Favorite Performance by Game Category

The previous section showed that, overall, rating favorites win more often than they lose. However, that result combines all game categories together.

The next question is whether rating advantage behaves the same way across Bullet, Blitz, Rapid, and Classical games. Since faster categories have more time pressure, rating advantage may become less reliable in those formats.

To test this, I group decisive games with a clear rating favorite by category. For each category, I calculate how often the favorite wins and how often an upset happens.

In this analysis, an upset means:

* White was the rating favorite, but Black won
* Black was the rating favorite, but White won

Close-rating games and draws are excluded from this section because there is no clear favorite winner to evaluate.


In [7]:
favorite_outcomes_by_category_query = """
WITH favorite_outcomes_by_category AS (
	SELECT 
		g.Category,
		COUNT(*) AS total_decisive_favorite_games,
		SUM(CASE
				WHEN (g.rating_favorite = 'White favorite' AND g.winner = 'White') 
				OR (g.rating_favorite = 'Black favorite' AND g.winner = 'Black') 
				THEN 1
				ELSE 0
			END) AS favorite_wins,
		SUM(CASE
				WHEN (g.rating_favorite = 'Black favorite' AND g.winner = 'White')
				OR (g.rating_favorite = 'White favorite' AND g.winner = 'Black')
				THEN 1
				ELSE 0
			END) AS upsets
	FROM games AS g
	WHERE g.winner IN ('White', 'Black')
	AND g.rating_favorite IN ('White favorite', 'Black favorite')
	GROUP BY g.Category
)

SELECT
	fobc.Category,
	fobc.total_decisive_favorite_games,
	fobc.favorite_wins,
	fobc.upsets,
	ROUND(fobc.favorite_wins * 100.0 / fobc.total_decisive_favorite_games, 2) AS favorite_win_rate,
	ROUND(fobc.upsets * 100.0 / fobc.total_decisive_favorite_games, 2) AS upset_rate
FROM favorite_outcomes_by_category AS fobc
ORDER BY favorite_win_rate DESC;
"""

favorite_outcomes_by_category_df = run_query(favorite_outcomes_by_category_query)

favorite_outcomes_by_category_df


,Category,total_decisive_favorite_games,favorite_wins,upsets,favorite_win_rate,upset_rate
0,Bullet,20790,13046,7744,62.75,37.25
1,Classical,11613,7192,4421,61.93,38.07
2,Rapid,19877,12297,7580,61.87,38.13
3,Blitz,35879,21470,14409,59.84,40.16


Across all categories, rating favorites win more often than they lose. The favorite win rate stays within a relatively narrow range, from about 60% in Blitz to about 63% in Bullet.

This suggests that rating advantage remains meaningful across different game categories. Faster categories may change how games end, especially through more time forfeits, but this result does not show a strong collapse of rating advantage in faster formats.

At the same time, upset rates remain fairly high across all categories, around 37–40%. This means that rating advantage is useful, but not overwhelming. A clear favorite is more likely to win, but underdog wins are still common.


## 6. Favorite Win Rate by Rating Gap Size

The previous section showed that rating favorites win at a fairly similar rate across game categories. This suggests that game category alone does not strongly change how reliable rating advantage is.

The next question is whether the **size** of the rating gap matters. A player who is 60 Elo points higher rated may not have the same advantage as a player who is 300 Elo points higher rated.

To test this, I group decisive games with a clear rating favorite into rating-gap buckets using `abs_rating_diff`. Then I calculate the favorite win rate and upset rate within each bucket.

This helps show whether rating advantage becomes more reliable as the Elo gap between players increases.


In [8]:
rating_gap_outcome_query = """
WITH rating_gap_outcomes AS (
    SELECT
        CASE
            WHEN abs_rating_diff BETWEEN 51 AND 99 THEN '51-99'
            WHEN abs_rating_diff BETWEEN 100 AND 199 THEN '100-199'
            WHEN abs_rating_diff BETWEEN 200 AND 399 THEN '200-399'
            ELSE '400+'
        END AS rating_gap_bucket,
        CASE
            WHEN abs_rating_diff BETWEEN 51 AND 99 THEN 1
            WHEN abs_rating_diff BETWEEN 100 AND 199 THEN 2
            WHEN abs_rating_diff BETWEEN 200 AND 399 THEN 3
            ELSE 4
        END AS bucket_order,
        COUNT(*) AS total_decisive_favorite_games,
        SUM(CASE
                WHEN (g.rating_favorite = 'White favorite' AND g.winner = 'White') 
                  OR (g.rating_favorite = 'Black favorite' AND g.winner = 'Black') 
                THEN 1
                ELSE 0
            END) AS favorite_wins,
        SUM(CASE
                WHEN (g.rating_favorite = 'Black favorite' AND g.winner = 'White')
                  OR (g.rating_favorite = 'White favorite' AND g.winner = 'Black')
                THEN 1
                ELSE 0
            END) AS upsets
    FROM games AS g
    WHERE g.winner IN ('White', 'Black')
        AND g.rating_favorite IN ('White favorite', 'Black favorite')
    GROUP BY rating_gap_bucket, bucket_order
)

SELECT
    rgo.rating_gap_bucket,
    rgo.bucket_order,
    rgo.total_decisive_favorite_games,
    rgo.favorite_wins,
    rgo.upsets,
    ROUND(rgo.favorite_wins * 100.0 / rgo.total_decisive_favorite_games, 2) AS favorite_win_rate,
    ROUND(rgo.upsets * 100.0 / rgo.total_decisive_favorite_games, 2) AS upset_rate
FROM rating_gap_outcomes AS rgo
ORDER BY rgo.bucket_order;
"""

rating_gap_outcome_df = run_query(rating_gap_outcome_query)

rating_gap_outcome_df

,rating_gap_bucket,bucket_order,total_decisive_favorite_games,favorite_wins,upsets,favorite_win_rate,upset_rate
0,51-99,1,40445,22619,17826,55.93,44.07
1,100-199,2,29600,18291,11309,61.79,38.21
2,200-399,3,13912,9782,4130,70.31,29.69
3,400+,4,4202,3313,889,78.84,21.16


The favorite win rate increases clearly as the rating gap becomes larger. In games where the rating gap is relatively small, between about 51 and 99 Elo points, the favorite wins only slightly more than half of the time. However, when the rating gap reaches 400 Elo points or more, the favorite wins almost 79% of decisive games.

This shows that rating advantage is not all-or-nothing. A small rating edge gives only a modest advantage, while a large rating gap makes the favorite much more reliable. This also explains why the overall favorite win rate was around 61%: it combines many small-gap games, where upsets are common, with fewer large-gap games, where favorites win much more often.


## 7. Favorite Win Rate by Rating Gap and Game Category

The previous section showed that the size of the rating gap strongly affects how often the favorite wins. Small rating gaps produced much lower favorite win rates, while very large rating gaps made the favorite much more reliable.

The next question is whether this rating-gap pattern behaves similarly across Bullet, Blitz, Rapid, and Classical games.

To test this, I group decisive games with a clear rating favorite by both `Category` and rating-gap bucket. This will allow me to compare favorite win rates across game categories while controlling for the size of the rating difference.

This is important because category-level differences alone can be misleading. If one category has more large rating gaps than another, its favorite win rate may look higher simply because the favorites are stronger on average. By comparing rating-gap buckets within each category, I can check whether time-control category still appears to matter after accounting for rating-gap size.


In [9]:
rating_gap_bucket_by_category_query = """
WITH category_rating_gap_outcomes AS (
	SELECT
		g.Category,
		CASE
			WHEN abs_rating_diff BETWEEN 51 AND 99 THEN '51-99'
			WHEN abs_rating_diff BETWEEN 100 AND 199 THEN '100-199'
			WHEN abs_rating_diff BETWEEN 200 AND 399 THEN '200-399'
			ELSE '400+'
		END AS rating_gap_bucket,
		COUNT(*) AS total_decisive_favorite_games,
		SUM(CASE
				WHEN (g.rating_favorite = 'White favorite' AND g.winner = 'White') 
				OR (g.rating_favorite = 'Black favorite' AND g.winner = 'Black') 
				THEN 1
				ELSE 0
			END) AS favorite_wins,
		SUM(CASE
				WHEN (g.rating_favorite = 'Black favorite' AND g.winner = 'White')
				OR (g.rating_favorite = 'White favorite' AND g.winner = 'Black')
				THEN 1
				ELSE 0
			END) AS upsets
	FROM games AS g
	WHERE g.winner IN ('White', 'Black')
		AND g.rating_favorite IN ('White favorite', 'Black favorite')
	GROUP BY 
		rating_gap_bucket,
		g.Category
	
)

SELECT
	crgo.Category,
	crgo.rating_gap_bucket,
	crgo.total_decisive_favorite_games,
	crgo.favorite_wins,
	crgo.upsets,
	ROUND(crgo.favorite_wins * 100.0 / crgo.total_decisive_favorite_games, 2) AS favorite_win_rate,
	ROUND(crgo.upsets * 100.0 / crgo.total_decisive_favorite_games, 2) AS upset_rate
FROM category_rating_gap_outcomes AS crgo
ORDER BY
    CASE crgo.rating_gap_bucket
        WHEN '400+' THEN 4
        WHEN '200-399' THEN 3
        WHEN '100-199' THEN 2
        WHEN '51-99' THEN 1
    END DESC,
    crgo.Category;
"""

rating_gap_bucket_by_category_df = run_query(rating_gap_bucket_by_category_query)

rating_gap_bucket_by_category_df


,Category,rating_gap_bucket,total_decisive_favorite_games,favorite_wins,upsets,favorite_win_rate,upset_rate
0,Blitz,400+,1554,1181,373,76.00,24.00
1,Bullet,400+,1324,1088,236,82.18,17.82
2,Classical,400+,328,245,83,74.70,25.30
3,Rapid,400+,996,799,197,80.22,19.78
4,Blitz,200-399,5317,3645,1672,68.55,31.45
5,Bullet,200-399,3551,2529,1022,71.22,28.78
6,Classical,200-399,1624,1180,444,72.66,27.34
7,Rapid,200-399,3420,2428,992,70.99,29.01
8,Blitz,100-199,11934,7242,4692,60.68,39.32
9,Bullet,100-199,6933,4292,2641,61.91,38.09


Across all game categories, the favorite win rate increases as the rating gap becomes larger. This confirms that rating-gap size is a strong driver of favorite performance.

The pattern is consistent across Bullet, Blitz, Rapid, and Classical games. In the smallest rating-gap bucket, favorites win only slightly more than half of decisive games. In the largest rating-gap bucket, favorites win much more often, usually around 75–82% of the time.

This suggests that the size of the rating difference matters more than the game category itself. Faster categories may affect how games end, especially through time forfeits, but the reliability of the rating favorite appears to be mainly explained by how large the Elo gap is.

Category-level differences should still be interpreted carefully. Some category-and-bucket combinations have fewer games, especially the `400+` bucket in Classical, so both the counts and percentages should be considered together.


## 8. Rating Gap Distribution by Game Category

The previous section showed that rating-gap size strongly affects how often the favorite wins. Before comparing favorite win rates across game categories, I should check whether the categories have similar rating-gap compositions.

If one category has more large rating-gap games than another, its favorite win rate could look higher simply because the favorites are stronger on average. This section checks how rating-gap buckets are distributed across Bullet, Blitz, Rapid, and Classical games.

To do this, I group decisive games with a clear rating favorite by `Category` and rating-gap bucket, then calculate both counts and percentages within each category.


In [10]:
rating_gap_distribution_category_query = """
WITH rating_gap_within_category AS (
	SELECT
		g.Category,
		CASE
            WHEN g.abs_rating_diff > 50 AND g.abs_rating_diff < 100 THEN '51-99'
            WHEN g.abs_rating_diff >= 100 AND g.abs_rating_diff < 200 THEN '100-199'
            WHEN g.abs_rating_diff >= 200 AND g.abs_rating_diff < 400 THEN '200-399'
			ELSE '400+'
		END AS rating_gap_bucket,
		COUNT(*) AS total_decisive_favorite_games
		FROM games AS g
		WHERE g.winner IN ('White', 'Black')
			AND g.rating_favorite IN ('White favorite', 'Black favorite')
		GROUP BY
			g.Category,
			rating_gap_bucket
)

SELECT
	rgwc.Category,
	rgwc.rating_gap_bucket,
	rgwc.total_decisive_favorite_games,
	ROUND(rgwc.total_decisive_favorite_games * 100.0 / SUM(rgwc.total_decisive_favorite_games) OVER (PARTITION BY rgwc.Category), 3) AS percentage_within_category
FROM rating_gap_within_category AS rgwc
ORDER BY
    rgwc.Category,
    CASE rgwc.rating_gap_bucket
        WHEN '400+' THEN 4
        WHEN '200-399' THEN 3
        WHEN '100-199' THEN 2
        WHEN '51-99' THEN 1
    END DESC;
"""

rating_gap_distribution_category_df = run_query(rating_gap_distribution_category_query)

rating_gap_distribution_category_df


,Category,rating_gap_bucket,total_decisive_favorite_games,percentage_within_category
0,Blitz,400+,1554,4.33
1,Blitz,200-399,5317,14.82
2,Blitz,100-199,11934,33.26
3,Blitz,51-99,17074,47.59
4,Bullet,400+,1324,6.37
5,Bullet,200-399,3551,17.08
6,Bullet,100-199,6933,33.35
7,Bullet,51-99,8982,43.20
8,Classical,400+,328,2.82
9,Classical,200-399,1624,13.98


## 9. Does Time Forfeit Change Rating Favorite Reliability?

The previous sections showed that rating favorites win more often as the rating gap becomes larger. This pattern was consistent across game categories, suggesting that rating-gap size is one of the strongest drivers of favorite performance.

The next question is whether this pattern changes when games end by time forfeit.

A time-forfeit ending may reflect a different kind of pressure than a normal ending. In normal games, the result may be more directly connected to chess strength. In time-forfeit games, the result may also depend on clock management, speed, and time pressure.

To test this, I will compare favorite win rates between `Normal` and `Time forfeit` games, while still grouping by rating-gap bucket. This helps check whether the rating favorite remains equally reliable when the game is decided by the clock.


In [11]:
rating_gap_bucket_by_termination_query = """
WITH termination_rating_gap_outcomes AS (
	SELECT
		g.Termination,
		CASE
			WHEN abs_rating_diff BETWEEN 51 AND 99 THEN '50-99'
			WHEN abs_rating_diff BETWEEN 100 AND 199 THEN '100-199'
			WHEN abs_rating_diff BETWEEN 200 AND 399 THEN '200-399'
			ELSE '400+'
		END AS rating_gap_bucket,
		COUNT(*) AS total_decisive_favorite_games,
		SUM(CASE
				WHEN (g.rating_favorite = 'White favorite' AND g.winner = 'White') 
				OR (g.rating_favorite = 'Black favorite' AND g.winner = 'Black') 
				THEN 1
				ELSE 0
			END) AS favorite_wins,
		SUM(CASE
				WHEN (g.rating_favorite = 'Black favorite' AND g.winner = 'White')
				OR (g.rating_favorite = 'White favorite' AND g.winner = 'Black')
				THEN 1
				ELSE 0
			END) AS upsets
	FROM games AS g
	WHERE g.winner IN ('White', 'Black')
		AND g.rating_favorite IN ('White favorite', 'Black favorite')
		AND g.Termination IN ('Normal', 'Time forfeit')
	GROUP BY 
		rating_gap_bucket,
		g.Termination
)

SELECT
	trgo.Termination,
	trgo.rating_gap_bucket,
	trgo.total_decisive_favorite_games,
	trgo.favorite_wins,
	trgo.upsets,
	ROUND(trgo.favorite_wins * 100.0 / trgo.total_decisive_favorite_games, 2) AS favorite_win_rate,
	ROUND(trgo.upsets * 100.0 / trgo.total_decisive_favorite_games, 2) AS upset_rate
FROM termination_rating_gap_outcomes AS trgo
ORDER BY
    CASE rating_gap_bucket
        WHEN '400+' THEN 4
        WHEN '200-399' THEN 3
        WHEN '100-199' THEN 2
        WHEN '51-99' THEN 1
    END DESC,
    CASE Termination
        WHEN 'Normal' THEN 1
        WHEN 'Time forfeit' THEN 2
    END;
"""

rating_gap_bucket_by_termination_df = run_query(rating_gap_bucket_by_termination_query)

rating_gap_bucket_by_termination_df


,Termination,rating_gap_bucket,total_decisive_favorite_games,favorite_wins,upsets,favorite_win_rate,upset_rate
0,Normal,400+,2948,2321,627,78.73,21.27
1,Time forfeit,400+,1253,991,262,79.09,20.91
2,Normal,200-399,10088,7086,3002,70.24,29.76
3,Time forfeit,200-399,3822,2695,1127,70.51,29.49
4,Normal,100-199,22053,13556,8497,61.47,38.53
5,Time forfeit,100-199,7545,4733,2812,62.73,37.27
6,Normal,50-99,30395,16925,13470,55.68,44.32
7,Time forfeit,50-99,10049,5693,4356,56.65,43.35


Within each rating-gap bucket, favorite win rates are very similar between `Normal` and `Time forfeit` games. This suggests that time-forfeit endings do not strongly reduce the reliability of rating advantage in this dataset.

The main driver of favorite performance still appears to be the size of the rating gap. Larger rating gaps produce higher favorite win rates regardless of whether the game ended normally or by time forfeit.

This adds nuance to the earlier findings. Faster categories have more time-forfeit endings, but those time-forfeit games do not appear to make rating favorites much less reliable. Clock pressure affects how games end, but rating gap size still explains much of who wins.


## 10. Comparing the Strength of Metadata Signals

The previous sections analyzed rating gap, game category, and termination type separately. However, not every pattern has the same analytical strength.

This section compares how much favorite win rate changes across different metadata factors. The goal is not to prove causation, but to identify which available metadata feature appears most strongly associated with favorite performance.

In particular, I compare:

- how much favorite win rate changes across rating-gap buckets
- how much favorite win rate changes across game categories within the same rating-gap bucket
- how much favorite win rate changes between normal and time-forfeit games within the same rating-gap bucket

This helps summarize whether rating gap, category, or termination type provides the strongest explanation for favorite reliability in the game-level metadata.


In [12]:
metadata_signal_strength_query = """
WITH base_favorite_games AS (
    SELECT
        g.Category,
        g.Termination,

        CASE
            WHEN g.abs_rating_diff > 50 AND g.abs_rating_diff < 100 THEN '51-99'
            WHEN g.abs_rating_diff >= 100 AND g.abs_rating_diff < 200 THEN '100-199'
            WHEN g.abs_rating_diff >= 200 AND g.abs_rating_diff < 400 THEN '200-399'
            ELSE '400+'
        END AS rating_gap_bucket,

        CASE
            WHEN (g.rating_favorite = 'White favorite' AND g.winner = 'White')
              OR (g.rating_favorite = 'Black favorite' AND g.winner = 'Black')
            THEN 1
            ELSE 0
        END AS favorite_won

    FROM games AS g
    WHERE g.winner IN ('White', 'Black')
      AND g.rating_favorite IN ('White favorite', 'Black favorite')
),

rating_gap_rates AS (
    SELECT
        bfg.rating_gap_bucket,
        COUNT(*) AS total_games,
        SUM(bfg.favorite_won) AS favorite_wins,
        SUM(bfg.favorite_won) * 100.0 / COUNT(*) AS favorite_win_rate
    FROM base_favorite_games AS bfg
    GROUP BY bfg.rating_gap_bucket
),

category_gap_rates AS (
    SELECT
        bfg.Category,
        bfg.rating_gap_bucket,
        COUNT(*) AS total_games,
        SUM(bfg.favorite_won) AS favorite_wins,
        SUM(bfg.favorite_won) * 100.0 / COUNT(*) AS favorite_win_rate
    FROM base_favorite_games AS bfg
    GROUP BY
        bfg.Category,
        bfg.rating_gap_bucket
),

termination_gap_rates AS (
    SELECT
        bfg.Termination,
        bfg.rating_gap_bucket,
        COUNT(*) AS total_games,
        SUM(bfg.favorite_won) AS favorite_wins,
        SUM(bfg.favorite_won) * 100.0 / COUNT(*) AS favorite_win_rate
    FROM base_favorite_games AS bfg
    WHERE bfg.Termination IN ('Normal', 'Time forfeit')
    GROUP BY
        bfg.Termination,
        bfg.rating_gap_bucket
),

signal_strength_summary AS (
    SELECT
        1 AS sort_order,
        0 AS bucket_order,
        'Across rating-gap buckets' AS comparison_type,
        'All clear-favorite games' AS comparison_group,
        MIN(rgr.favorite_win_rate) AS min_favorite_win_rate,
        MAX(rgr.favorite_win_rate) AS max_favorite_win_rate,
        MAX(rgr.favorite_win_rate) - MIN(rgr.favorite_win_rate) AS range_percentage_points
    FROM rating_gap_rates AS rgr

    UNION ALL

    SELECT
        2 AS sort_order,
        CASE cgr.rating_gap_bucket
            WHEN '400+' THEN 4
            WHEN '200-399' THEN 3
            WHEN '100-199' THEN 2
            WHEN '51-99' THEN 1
        END AS bucket_order,
        'Across categories within rating-gap bucket' AS comparison_type,
        cgr.rating_gap_bucket AS comparison_group,
        MIN(cgr.favorite_win_rate) AS min_favorite_win_rate,
        MAX(cgr.favorite_win_rate) AS max_favorite_win_rate,
        MAX(cgr.favorite_win_rate) - MIN(cgr.favorite_win_rate) AS range_percentage_points
    FROM category_gap_rates AS cgr
    GROUP BY cgr.rating_gap_bucket

    UNION ALL

    SELECT
        3 AS sort_order,
        CASE tgr.rating_gap_bucket
            WHEN '400+' THEN 4
            WHEN '200-399' THEN 3
            WHEN '100-199' THEN 2
            WHEN '51-99' THEN 1
        END AS bucket_order,
        'Normal vs time forfeit within rating-gap bucket' AS comparison_type,
        tgr.rating_gap_bucket AS comparison_group,
        MIN(tgr.favorite_win_rate) AS min_favorite_win_rate,
        MAX(tgr.favorite_win_rate) AS max_favorite_win_rate,
        MAX(tgr.favorite_win_rate) - MIN(tgr.favorite_win_rate) AS range_percentage_points
    FROM termination_gap_rates AS tgr
    GROUP BY tgr.rating_gap_bucket
)

SELECT
    sss.comparison_type,
    sss.comparison_group,
    ROUND(sss.min_favorite_win_rate, 2) AS min_favorite_win_rate,
    ROUND(sss.max_favorite_win_rate, 2) AS max_favorite_win_rate,
    ROUND(sss.range_percentage_points, 2) AS range_percentage_points
FROM signal_strength_summary AS sss
ORDER BY
    sss.sort_order,
    sss.bucket_order DESC;
"""

metadata_signal_strength_df = run_query(metadata_signal_strength_query)

metadata_signal_strength_df

,comparison_type,comparison_group,min_favorite_win_rate,max_favorite_win_rate,range_percentage_points
0,Across rating-gap buckets,All clear-favorite games,55.93,78.84,22.92
1,Across categories within rating-gap bucket,400+,74.70,82.18,7.48
2,Across categories within rating-gap bucket,200-399,68.55,72.66,4.11
3,Across categories within rating-gap bucket,100-199,60.68,63.82,3.14
4,Across categories within rating-gap bucket,51-99,55.07,57.19,2.13
5,Normal vs time forfeit within rating-gap bucket,400+,78.73,79.09,0.36
6,Normal vs time forfeit within rating-gap bucket,200-399,70.24,70.51,0.27
7,Normal vs time forfeit within rating-gap bucket,100-199,61.47,62.73,1.26
8,Normal vs time forfeit within rating-gap bucket,51-99,55.68,56.65,0.97


The comparison shows that rating-gap size is the strongest metadata signal for favorite reliability.

Across rating-gap buckets, the favorite win rate changes by about 22.9 percentage points, from roughly 56% in the smallest rating-gap bucket to almost 79% in the largest bucket. This is a much larger difference than the category-level differences within the same rating-gap bucket.

Game category still shows some variation, especially in the `400+` rating-gap bucket, where the range across categories is about 7.5 percentage points. However, most category-level ranges are much smaller than the overall rating-gap effect.

Termination type appears to have the weakest effect in this comparison. Within the same rating-gap bucket, the favorite win rate is very similar between `Normal` and `Time forfeit` games, with differences of only about 0.3 to 1.3 percentage points.

Overall, this supports the main metadata-level conclusion: game category strongly affects how games end, especially through time forfeits, but rating-gap size is the clearest driver of whether the favorite wins. Time-forfeit endings do not appear to meaningfully weaken rating advantage once rating gap is considered.


## 11. SQL Findings Summary and Transition to Engine Evaluation Analysis

This notebook used SQL to analyze the clean game-level Lichess dataset created in the feasibility notebook. The analysis focused on metadata-level variables such as game category, termination type, winner, rating favorite, and rating gap size.

The main finding is that game category strongly affects **how games end**. Bullet games have the highest time-forfeit rate, while Classical games have the lowest. This suggests that faster game formats are much more affected by clock pressure.

However, time-forfeit wins are almost evenly split between White and Black, so the higher time-forfeit rate does not appear to create a strong color advantage.

Rating advantage also matters, but its strength depends heavily on the size of the rating gap. Clear rating favorites win more often than they lose, but small rating gaps produce much lower favorite win rates than large rating gaps.

The final synthesis showed that rating-gap size is the strongest metadata signal for favorite reliability. Favorite win rate changed by about 22.9 percentage points across rating-gap buckets, while category-level differences within the same rating-gap bucket were smaller. Differences between normal and time-forfeit games were even smaller.

Overall, the metadata-level conclusion is:

**Game category strongly affects how games end, especially through time forfeits, but rating-gap size is the clearest metadata driver of whether the favorite wins. Time-forfeit endings do not appear to meaningfully weaken rating advantage once rating gap is considered.**

This gives a solid game-level understanding of the dataset, but it also shows the limits of metadata analysis. Metadata can describe who played, who won, and how the game ended, but it cannot explain the actual chess events inside the game.

The next notebook will move into engine-evaluation features to explore deeper chess-specific patterns, such as evaluation swings, possible blunders, and turning points.
